In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 


In [3]:
### Ler o documento
def read_file(path):
    with open(path, 'r') as archive:
        lines = archive.read().splitlines()
        lines.pop(0)
        return lines
        
### Encontrar as linhas de cada linha atômica
def new_specline(self):
    return "/TYPE" in self and "A" in self

### Gerar o dicionário com as temperaturas e densidades para cada comprimento de onda
def get_pec(self):
    wavelength = []
    transition_type = []
    
    pec = []
    # Indice no qual iremos iterar
    idx = 0
    length_data = len(self)

    ### Iterar sobre todas as linhas
    while idx < length_data:
        
        row = self[idx]
        
        ### Checar se linha é comprimento de onda novo (True se sim e False se não)
        if new_specline(row):
            match = re.search(r"(\d+\.\d+)A",row) ### Encontra o comprimento de onda
            
            ### Se o comprimento de onda estiver ali pula as linhas até atingir as pecs
            if match:
                
                wavelength.append(float(match.group(1))) # Pega o comprimento de onda em [A]
                match_type = re.search(r'/TYPE\s=\s(\S+)',row) # Encontra o tipo de transição
                transition_type.append(match_type.group(1)) # Salva o tipo de transição.
                
                ###Pular as lihas
                idx += 7
            
                ### Define as listas de valores e a lista que servirá como matriz
                values = []
                pec_for_wavelength = [] 

                ### Itera sobre os blocos de 3 em 3 linhas para criar as linhas:
                for _ in range(int(3 * 24)):
                    if idx < length_data:
                        row_data = self[idx]
                        
                        # Separa os valores tratadas pela notação científica do FORTRAN (D -> E):
                        for num in row_data.replace('D', 'E').split():
                            values.append(float(num))

                        idx += 1
                        
                pec_for_wavelength = np.array(values).reshape(24, 21)   
                
                ### Adiciona as matrizes para cada comprimento de onda dentro da matriz pec
                pec.append(pec_for_wavelength)

                ### Continue apenas para não bugar o loop? (gepeto ajudou)
                continue
                
        ### Dicionário contendo os comprimentos de onda, o tipo de transição e as matrizes correspondentes)       
        dic = {
            r"Comprimento de Onda": wavelength,
            r"Transition_type":  transition_type,
            r"Matriz_pec": pec
        }

                
        idx += 1
    ### Retorna um dicionário com matrizes de dimensão (24x21) - (N_densidades x N_temperaturas)
    return dic